## 03. Binary + Masked Shade Ablation Training

This notebook runs an isolated ablation using the same split manifests from 02:
- Inputs: `data/processed/splits/{train,val,test}.csv`
- Backbone: EfficientNetB0 (ImageNet weights if available)
- Heads:
  - `bin_head`: sigmoid for core binary labels (`*_p`)
  - `shade_head`: 2-class softmax, trained with conditional mask from `walking_paths_p`
- Excludes `score_head` and `veg_head` for this ablation.


In [2]:
# Imports and paths
import os
import random
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# Global reproducibility controls
GLOBAL_SEED = 123
RNG_STATE_AUG = 123

# Set seeds
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

train_csv = Path('../data/processed/splits/train.csv')
val_csv   = Path('../data/processed/splits/val.csv')
test_csv  = Path('../data/processed/splits/test.csv')

assert train_csv.exists() and val_csv.exists() and test_csv.exists(), 'Missing split manifests. Run 02 first.'

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

print('Loaded splits:', len(train_df), len(val_df), len(test_df))


Loaded splits: 2347 782 782


In [3]:
# Build tf.data datasets from manifests (binary + masked shade ablation)
IMG_SIZE = (512, 512)
BATCH_SIZE = 8

EXPERIMENT_MODE = 'binary_plus_shade_masked'

# Toggle augmentation per run
USE_AUGMENTATION = False
print('EXPERIMENT_MODE =', EXPERIMENT_MODE)
print('USE_AUGMENTATION =', USE_AUGMENTATION)

# Binary labels are stored as probabilities in *_p columns
binary_cols = [c for c in train_df.columns if c.endswith('_p')]
HAS_SHADE_CLASS = 'shade_class' in train_df.columns

assert 'image_path' in train_df.columns, "Missing image_path in split manifests"
assert len(binary_cols) > 0, "No binary *_p columns found in split manifests"
assert HAS_SHADE_CLASS, "Missing shade_class in split manifests"

print('Binary labels:', binary_cols)
print('Using shade_class:', HAS_SHADE_CLASS)

NUM_SHADE = 2  # minimal vs abundant
SHADE_LOSS_MODE = 'sparse'

# Map a row to (image, label dict)
def decode_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32) / 255.0
    return img

# Geometry-only augmentation (safe for color-sensitive labels)
def augment_geom(img):
    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    return img

# Shade is only meaningful when paths exist.
# We mask shade loss using ground-truth walking_paths (soft) so walking_paths=0 contributes ~0.
def get_shade_sample_weight(df):
    if 'walking_paths_p' in df.columns:
        return df['walking_paths_p'].fillna(0.0).astype(np.float32).values
    if 'walking_paths' in df.columns:
        return df['walking_paths'].fillna(0).astype(np.float32).values
    raise ValueError(
        "Cannot compute shade mask: need walking_paths_p (preferred) or walking_paths in split manifests"
    )

# Build a dataset from a DataFrame
# - Apply augmentation only on TRAIN when USE_AUGMENTATION=True
# - Never augment val/test
# - Cache decoded images to reduce disk I/O; augmentation stays after cache
# - Emit per-output sample weights to mask shade loss
def make_ds(df, shuffle=True, augment=False):
    paths = df['image_path'].astype(str).tolist()

    y_bin = df[binary_cols].fillna(0.0).astype(np.float32).values
    y_shade = df['shade_class'].fillna(0).astype(np.int32).values
    y_shade = np.clip(y_shade, 0, NUM_SHADE - 1)

    shade_sw = get_shade_sample_weight(df)
    ones = np.ones((len(paths),), dtype=np.float32)

    ds_paths = tf.data.Dataset.from_tensor_slices(paths)
    ds_imgs = ds_paths.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)

    ds_labels = tf.data.Dataset.from_tensor_slices({
        'bin_head': y_bin,
        'shade_head': y_shade,
    })

    ds_sw = tf.data.Dataset.from_tensor_slices({
        'bin_head': ones,
        'shade_head': shade_sw,
    })

    ds = tf.data.Dataset.zip((ds_imgs, ds_labels, ds_sw))

    if shuffle and len(paths) > 1:
        ds = ds.shuffle(buffer_size=len(paths), seed=GLOBAL_SEED, reshuffle_each_iteration=True)

    ds = ds.cache()

    if augment:
        ds = ds.map(lambda img, y, sw: (augment_geom(img), y, sw), num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train_df, shuffle=True, augment=USE_AUGMENTATION)
val_ds   = make_ds(val_df, shuffle=False, augment=False)
test_ds  = make_ds(test_df, shuffle=False, augment=False)

print('Shade mask mean (train/val/test):',
      float(np.mean(get_shade_sample_weight(train_df))),
      float(np.mean(get_shade_sample_weight(val_df))),
      float(np.mean(get_shade_sample_weight(test_df))))
print('Datasets ready.')


EXPERIMENT_MODE = binary_plus_shade_masked
USE_AUGMENTATION = False
Binary labels: ['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'gardens_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']
Using shade_class: True
Shade mask mean (train/val/test): 0.7281990647315979 0.7291133999824524 0.7329496741294861
Datasets ready.


In [ ]:
# Optional: save a small augmentation preview grid
# This helps verify augmentations are label-preserving and not too aggressive.

if not USE_AUGMENTATION:
    print('Augmentation disabled (USE_AUGMENTATION=False). Skipping preview.')
else:
    from datetime import datetime
    import matplotlib.pyplot as plt

    out_dir = Path('../data/interim/aug_preview').resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = globals().get('RUN_TAG', None) or datetime.now().strftime('%Y%m%d_%H%M%S')

    # train_ds yields (img, labels, sample_weight)
    batch_imgs, _, _ = next(iter(train_ds.take(1)))
    n = min(int(batch_imgs.shape[0]), 8)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()
    for i in range(8):
        axes[i].axis('off')
    for i in range(n):
        axes[i].imshow(batch_imgs[i].numpy())
        axes[i].set_title(f"aug {i}")
        axes[i].axis('off')

    plt.tight_layout()
    out_path = out_dir / f"aug_preview_binary_ablation_{tag}.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print('Saved augmentation preview to', out_path)


In [ ]:
# Define a binary+shade model (EfficientNetB0 backbone)
from tensorflow.keras import layers, models, applications, optimizers

tf.keras.backend.clear_session()

INPUT_SHAPE = (512, 512, 3)
print('INPUT_SHAPE used for backbone:', INPUT_SHAPE)
assert INPUT_SHAPE[-1] == 3, f"Expected 3-channel RGB input for imagenet weights, got {INPUT_SHAPE}"

NUM_BIN = len(binary_cols)
assert NUM_SHADE == 2, NUM_SHADE

# Explicit input tensor (forces 3-channel model build)
inputs = layers.Input(shape=INPUT_SHAPE, name='img')

# Step 1: sanity-build WITHOUT weights
backbone_no_weights = applications.EfficientNetB0(include_top=False, weights=None, input_tensor=inputs)
stem0 = backbone_no_weights.get_layer('stem_conv')
print('Sanity stem_conv kernel shape (weights=None):', tuple(stem0.kernel.shape))
assert int(stem0.kernel.shape[2]) == 3

# Step 2: try ImageNet weights
try:
    backbone = applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
    stem = backbone.get_layer('stem_conv')
    print('Loaded ImageNet weights. stem_conv kernel shape:', tuple(stem.kernel.shape))
except Exception as e:
    print('FAILED to load ImageNet weights for EfficientNetB0:', repr(e))
    print('Falling back to weights=None so you can proceed with training.')
    backbone = backbone_no_weights

x = layers.GlobalAveragePooling2D()(backbone.output)

# Heads (ablation: binary + shade only)
bin_out = layers.Dense(NUM_BIN, activation='sigmoid', name='bin_head')(x)
shade_out = layers.Dense(NUM_SHADE, activation='softmax', name='shade_head')(x)

model = models.Model(inputs=inputs, outputs={
    'bin_head': bin_out,
    'shade_head': shade_out,
})
print('Model outputs:', model.output_names)
assert set(model.output_names) == {'bin_head', 'shade_head'}, model.output_names

# Binary head loss mode:
# - focal: often improves PR-AUC / recall on rare labels
# - bce  : standard binary cross-entropy baseline
USE_FOCAL_LOSS = False  # toggle per run
print('Binary loss mode:', 'focal' if USE_FOCAL_LOSS else 'binary_crossentropy')

# ---- Focal loss components (kept in-place even if USE_FOCAL_LOSS=False) ----
bin_prev = train_df[binary_cols].fillna(0.0).astype(np.float32).mean().values
alpha_vec = tf.constant(np.clip(1.0 - bin_prev, 0.25, 0.95), dtype=tf.float32)
FOCAL_GAMMA = 2.0
_EPS = 1e-7

bin_names = [c[:-2] for c in binary_cols]
if USE_FOCAL_LOSS:
    print('Binary focal alpha per label:')
    for n, a, p in zip(bin_names, alpha_vec.numpy().tolist(), bin_prev.tolist()):
        print(f"  {n:24s} prevalence={p:.3f} alpha={a:.3f}")

def bin_focal_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), _EPS, 1.0 - _EPS)
    alpha_t = y_true * alpha_vec + (1.0 - y_true) * (1.0 - alpha_vec)
    p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
    focal = -alpha_t * tf.pow(1.0 - p_t, FOCAL_GAMMA) * tf.math.log(p_t)
    return tf.reduce_mean(tf.reduce_mean(focal, axis=-1))

bin_loss = bin_focal_loss if USE_FOCAL_LOSS else 'binary_crossentropy'

losses = {
    'bin_head': bin_loss,
    'shade_head': 'sparse_categorical_crossentropy',
}
metrics = {
    'bin_head': [
        'binary_accuracy',
        tf.keras.metrics.AUC(curve='PR', multi_label=True, name='pr_auc'),
    ],
    'shade_head': ['sparse_categorical_accuracy'],
}

loss_weights_binary_shade = {
    'bin_head': 1.0,
    'shade_head': 1.0,
}

model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss=losses,
    metrics=metrics,
    loss_weights=loss_weights_binary_shade,
)

model.summary()


In [5]:
# Train (warm-up then fine-tune)
from datetime import datetime

# One tag per run so artifacts don't overwrite each other.
RUN_TAG = globals().get('RUN_TAG', None) or datetime.now().strftime('%Y%m%d_%H%M%S')
print('RUN_TAG:', RUN_TAG)

EPOCHS_WARMUP = 5
EPOCHS_FINETUNE = 100

# Warm-up: freeze backbone, train heads
for layer in model.layers:
    if isinstance(layer, tf.keras.Model) or layer.name.startswith('efficientnet'):
        layer.trainable = False

# Save artifacts under models/runs/binary_ablation_<RUN_TAG>/
run_dir = Path('../models/runs') / f'binary_ablation_{RUN_TAG}'
run_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = run_dir / f'best_binary_ablation_{RUN_TAG}.keras'

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ckpt_path),
        monitor='val_bin_head_pr_auc',
        mode='max',
        save_best_only=True,
        save_weights_only=False,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_bin_head_pr_auc',
        mode='max',
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_bin_head_pr_auc',
        mode='max',
        factor=0.5,
        patience=2,
    ),
]

# Sanity checks: ensure the selected binary loss mode is in effect
if USE_FOCAL_LOSS:
    assert callable(losses.get('bin_head', None)), f"Expected focal loss function for bin_head, got: {losses.get('bin_head')}"
else:
    assert losses.get('bin_head', None) == 'binary_crossentropy', f"Expected 'binary_crossentropy' for bin_head, got: {losses.get('bin_head')}"

print('Sanity: USE_FOCAL_LOSS =', USE_FOCAL_LOSS)
print('Sanity: bin_head loss =', losses['bin_head'])

""" # Debug one batch to ensure labels/sample weights are keyed by output name.
_, y_dbg, sw_dbg = next(iter(train_ds.take(1)))
assert isinstance(y_dbg, dict) and isinstance(sw_dbg, dict), (type(y_dbg), type(sw_dbg))
assert set(y_dbg.keys()) == {'bin_head', 'shade_head'}, y_dbg.keys()
assert set(sw_dbg.keys()) == {'bin_head', 'shade_head'}, sw_dbg.keys()
print('Batch label keys:', list(y_dbg.keys()), '| sample_weight keys:', list(sw_dbg.keys())) """

history_warmup = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_WARMUP,
    callbacks=callbacks,
    verbose=1,
)

# Fine-tune: unfreeze backbone
for layer in model.layers:
    layer.trainable = True

# Sanity checks again before fine-tune compile
if USE_FOCAL_LOSS:
    assert callable(losses.get('bin_head', None)), f"Expected focal loss function for bin_head, got: {losses.get('bin_head')}"
else:
    assert losses.get('bin_head', None) == 'binary_crossentropy', f"Expected 'binary_crossentropy' for bin_head, got: {losses.get('bin_head')}"

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=losses,
    metrics=metrics,
    loss_weights=loss_weights_binary_shade,
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=callbacks,
    verbose=1,
)

print('Training complete.')


RUN_TAG: 20260223_201201
Sanity: USE_FOCAL_LOSS = False
Sanity: bin_head loss = binary_crossentropy
Epoch 1/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 521s 2s/step - bin_head_binary_accuracy: 0.8120 - bin_head_loss: 0.3975 - bin_head_pr_auc: 0.5549 - loss: 0.9011 - shade_head_loss: 0.5029 - shade_head_sparse_categorical_accuracy: 0.6336 - val_bin_head_binary_accuracy: 0.6308 - val_bin_head_loss: 1.1248 - val_bin_head_pr_auc: 0.3603 - val_loss: 1.6146 - val_shade_head_loss: 0.4883 - val_shade_head_sparse_categorical_accuracy: 0.7046 - learning_rate: 0.0010
Epoch 2/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 515s 2s/step - bin_head_binary_accuracy: 0.8332 - bin_head_loss: 0.3590 - bin_head_pr_auc: 0.6107 - loss: 0.8046 - shade_head_loss: 0.4448 - shade_head_sparse_categorical_accuracy: 0.6549 - val_bin_head_binary_accuracy: 0.6308 - val_bin_head_loss: 0.9077 - val_bin_head_pr_auc: 0.4168 - val_loss: 1.3956 - val_shade_head_loss: 0.4867 - val_shade_head_sparse_categorical_accuracy: 0.7046 - learning_rate: 0.0010


In [6]:
# Evaluation moved to 04_model_evaluation.ipynb
print('Note: Evaluation and threshold calibration are now in 04_model_evaluation.ipynb')


Note: Evaluation and threshold calibration are now in 04_model_evaluation.ipynb


In [6]:
# Save final artifacts: trained model and config
from datetime import datetime
import json

# Reuse the same RUN_TAG used for checkpoints (or make one if needed)
RUN_TAG = globals().get('RUN_TAG', None) or datetime.now().strftime('%Y%m%d_%H%M%S')

# Save artifacts under models/runs/binary_ablation_<RUN_TAG>/
save_dir = Path('../models/runs') / f'binary_ablation_{RUN_TAG}'
save_dir.mkdir(parents=True, exist_ok=True)

# 1) Save final model
final_path = save_dir / f"final_binary_ablation_{RUN_TAG}.keras"
model.save(str(final_path))
print('Saved final model to', final_path)

# 2) Save weights separately
weights_path = save_dir / f"final_binary_ablation_{RUN_TAG}.weights.h5"
model.save_weights(str(weights_path))
print('Saved weights to', weights_path)

# 3) Save label/config metadata
config = {
    'experiment_mode': EXPERIMENT_MODE,
    'run_tag': RUN_TAG,
    'img_size': IMG_SIZE,
    'binary_cols': binary_cols,
    'shade_class_col': 'shade_class',
    'num_bin': int(len(binary_cols)),
    'num_shade': int(NUM_SHADE),
    'shade_mask_source': 'walking_paths_p',
}

config_path = save_dir / f"model_config_binary_ablation_{RUN_TAG}.json"
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print('Saved', config_path.name)


Saved final model to ../models/runs/binary_ablation_20260223_201201/final_binary_ablation_20260223_201201.keras
Saved weights to ../models/runs/binary_ablation_20260223_201201/final_binary_ablation_20260223_201201.weights.h5
Saved model_config_binary_ablation_20260223_201201.json
